# OBR — Fase Verde 3: segmentação neural dos marcadores

Compara LinhaNet e LR-ASPP, calibra o limiar somente na validação e gera um único ZIP final. O teste fechado e os 271 casos de active learning não fazem parte do pacote. Selecione uma GPU T4 em **Ambiente de execução → Alterar tipo de ambiente de execução** e execute todas as células.

In [ ]:
import subprocess
import sys
from pathlib import Path

REVISAO_CODIGO = "05dedb72aa25be6372f9cc74b58ef30f7f1c147f"
REPOSITORIO = Path("/content/OBR")
if not REPOSITORIO.exists():
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/DaviBonetto/OBR.git", str(REPOSITORIO)],
        check=True,
    )
subprocess.run(["git", "-C", str(REPOSITORIO), "checkout", "--detach", REVISAO_CODIGO], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPOSITORIO}[treinamento]"], check=True
)
print(
    "Código fixado em:",
    subprocess.check_output(
        ["git", "-C", str(REPOSITORIO), "rev-parse", "HEAD"], text=True
    ).strip(),
)

In [ ]:
import hashlib
import json
import shutil

from google.colab import files

NOME_PACOTE = "fase_verde_3_dataset_v1.zip"
HASH_ESPERADO = "c2d1badc4dd8224c06a186dad7ce5264ccb7b3996917b06bd1c3332e649042ec"
TAMANHO_ESPERADO = 501481335
print(f"Selecione {NOME_PACOTE} (aprox. 502 MB).")
enviados = files.upload()
assert list(enviados) == [NOME_PACOTE], "Envie somente o ZIP solicitado."
PACOTE = Path.cwd() / NOME_PACOTE
assert PACOTE.stat().st_size == TAMANHO_ESPERADO, PACOTE.stat().st_size
digest = hashlib.sha256()
with PACOTE.open("rb") as arquivo:
    for bloco in iter(lambda: arquivo.read(1024 * 1024), b""):
        digest.update(bloco)
hash_obtido = digest.hexdigest()
assert hash_obtido == HASH_ESPERADO, (hash_obtido, HASH_ESPERADO)

In [ ]:
DATASET = Path("/content/fase_verde_3_dataset_v1")
if DATASET.exists():
    shutil.rmtree(DATASET)
shutil.unpack_archive(PACOTE, DATASET)
manifesto = json.loads((DATASET / "manifesto.json").read_text())
assert manifesto["divisao_teste_incluida"] is False
assert manifesto["quantidades"] == {"total": 2085, "treino": 1302, "validacao": 783}
indice = [
    json.loads(linha) for linha in (DATASET / "indice.jsonl").read_text().splitlines() if linha
]
assert len(indice) == 2085
assert {item["divisao"] for item in indice} == {"treino", "validacao"}
assert all("categoria_verde" in item for item in indice)
print("Dataset íntegro; teste ausente:", manifesto["quantidades"])

In [ ]:
import torch

assert torch.cuda.is_available(), "Ative a GPU T4 antes do treinamento."
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)

## Treinamento das duas arquiteturas
A LinhaNet mede o limite de velocidade; a LR-ASPP mede o limite de precisão. Nenhuma delas é promovida sem passar pelos gates de validação e pela inspeção posterior.

In [ ]:
RESULTADOS = Path("/content/resultados_verde_fase3")
if RESULTADOS.exists():
    shutil.rmtree(RESULTADOS)
CONFIGURACAO = REPOSITORIO / "configuracoes/treinamento_verde_fase3.toml"
experimentos = {
    "linhanet_verde_v1": "linhanet",
    "lraspp_verde_v1": "lraspp_mobilenet_v3_large",
}
for nome, arquitetura in experimentos.items():
    print("Iniciando:", nome)
    subprocess.run(
        [
            "obr-treinar-segmentacao",
            "--dataset",
            str(DATASET),
            "--saida",
            str(RESULTADOS / nome),
            "--configuracao",
            str(CONFIGURACAO),
            "--arquitetura",
            arquitetura,
            "--epocas",
            "100",
            "--lote",
            "32",
            "--trabalhadores",
            "2",
            "--paciencia",
            "18",
        ],
        check=True,
    )

## Calibração exclusiva na validação
Cada checkpoint é executado uma vez na validação para comparar limiares. O teste continua inacessível.

In [ ]:
from obr_oficial.treinamento.segmentacao import avaliar_limiares_checkpoint

limiares = [valor / 100 for valor in range(30, 96, 5)]
avaliacoes = {}
for nome in experimentos:
    avaliacao = avaliar_limiares_checkpoint(DATASET, RESULTADOS / nome / "melhor.pt", limiares)
    avaliacoes[nome] = avaliacao
    (RESULTADOS / nome / "calibracao_limiares.json").write_text(
        json.dumps(avaliacao, ensure_ascii=False, indent=2)
    )
    print(nome, max(avaliacao["resultados"], key=lambda item: item["dice"]))

In [ ]:
GATES = {"dice": 0.95, "precisao": 0.97, "recall": 0.93, "fpr_significativo": 0.05}
candidatos = []
for nome, avaliacao in avaliacoes.items():
    for metricas in avaliacao["resultados"]:
        passou = (
            metricas["dice"] >= GATES["dice"]
            and metricas["precisao"] >= GATES["precisao"]
            and metricas["recall"] >= GATES["recall"]
            and metricas["taxa_falso_positivo_negativos_significativos"]
            <= GATES["fpr_significativo"]
        )
        pontuacao = (
            metricas["dice"] - 0.40 * metricas["taxa_falso_positivo_negativos_significativos"]
        )
        candidatos.append(
            {"experimento": nome, "passou": passou, "pontuacao": pontuacao, **metricas}
        )
aprovados = [item for item in candidatos if item["passou"]]
vencedor = max(aprovados or candidatos, key=lambda item: item["pontuacao"])
comparacao = {
    "teste_aberto": False,
    "gates": GATES,
    "algum_candidato_passou": bool(aprovados),
    "vencedor_provisorio": vencedor,
    "candidatos": candidatos,
}
(RESULTADOS / "comparacao.json").write_text(json.dumps(comparacao, ensure_ascii=False, indent=2))
print(json.dumps(comparacao["vencedor_provisorio"], indent=2))

## Entrega única
Baixa `OBR_VERDE_FASE3_RESULTADOS_T4.zip`. Envie somente esse arquivo ao Codex para auditoria, visualização e preparação do active learning.

In [ ]:
import datetime
import platform

import torchvision

ENTREGA = Path("/content/OBR_VERDE_FASE3_RESULTADOS_T4")
if ENTREGA.exists():
    shutil.rmtree(ENTREGA)
shutil.copytree(RESULTADOS, ENTREGA / "resultados")
shutil.copy2(CONFIGURACAO, ENTREGA)
shutil.copy2(REPOSITORIO / "dados/manifestos/fase_verde_3_dataset_v1.json", ENTREGA)
ambiente = {
    "gerado_utc": datetime.datetime.now(datetime.UTC).isoformat(),
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0),
    "git_commit": REVISAO_CODIGO,
    "sha256_dataset": hash_obtido,
    "teste_aberto": False,
}
(ENTREGA / "ambiente.json").write_text(json.dumps(ambiente, ensure_ascii=False, indent=2))
hashes = {}
for caminho in sorted(ENTREGA.rglob("*")):
    if caminho.is_file():
        hashes[caminho.relative_to(ENTREGA).as_posix()] = hashlib.sha256(
            caminho.read_bytes()
        ).hexdigest()
(ENTREGA / "sha256_arquivos.json").write_text(json.dumps(hashes, indent=2, sort_keys=True))
ARQUIVO_FINAL = Path(shutil.make_archive("/content/OBR_VERDE_FASE3_RESULTADOS_T4", "zip", ENTREGA))
print("ZIP final:", ARQUIVO_FINAL)
print("SHA-256:", hashlib.sha256(ARQUIVO_FINAL.read_bytes()).hexdigest())
files.download(str(ARQUIVO_FINAL))